## Read json 1

In [1]:
pip install emoji

Note: you may need to restart the kernel to use updated packages.


In [2]:
import emoji.unicode_codes
import pandas as pd
import json
import ast
import re
import emoji

emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002500-\U00002BEF"  # chinese char
                               u"\U00002702-\U000027B0"
                               u"\U00002702-\U000027B0"
                               u"\U000024C2-\U0001F251"
                               u"\U0001f926-\U0001f937"
                               u"\U00010000-\U0010ffff"
                               u"\u2640-\u2642"
                               u"\u2600-\u2B55"
                               u"\u200d"
                               u"\u23cf"
                               u"\u23e9"
                               u"\u231a"
                               u"\ufe0f"  # dingbats
                               u"\u3030"
                               "]+", flags=re.UNICODE)


def remove_price_bitcoin(text):
    # Regex pattern to match "$<number> Bitcoin"
    pattern = r"\$\d{1,3}(?:,\d{3})*(?:\.\d+)? Bitcoin"
    cleaned_text = re.sub(pattern, "", text, flags=re.IGNORECASE).strip()
    pattern = r"Bitcoin falls under \$\d{1,3}(?:,\d{3})*(?:\.\d+)?."
    cleaned_text = re.sub(pattern, "", cleaned_text, flags=re.IGNORECASE).strip()
    # pattern = r"\$\d{1,3}(?:,\d{3})*(?:\.\d+)?"
    # cleaned_text = re.sub(pattern, "", cleaned_text, flags=re.IGNORECASE).strip()
    return cleaned_text


def clean_text_column(text):
    try:
        # Convert string representation of list to actual list
        data = ast.literal_eval(text) if isinstance(text, str) else text
        
        if not isinstance(data, list):
            return text  # Return as is if it's not a list
        
        # Remove 'mention' type elements
        cleaned_data = [
            item for item in data
            if not (isinstance(item, dict) and item.get("type") == "mention")
            if not (isinstance(item, dict) and item.get("type") == "text_link")
            if not (isinstance(item, dict) and item.get("type") == "bold")
        ]
        
        # Remove newline characters
        cleaned_data = [item.replace("\n", "") if isinstance(item, str) else "" for item in cleaned_data]


        merged_text = " ".join(cleaned_data).strip()
        cleaned_text = emoji_pattern.sub(r'', merged_text ) #emoji.demojize(merged_text)
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()  # Remove extra spaces
        cleaned_text = remove_price_bitcoin(cleaned_text)

        
        return cleaned_text # Convert back to string if needed
    
    except (SyntaxError, ValueError):
        return text  # Return as is if conversion fails



In [3]:
file = 'result.json'
with open(file) as watcher_guru:
    data = json.load(watcher_guru)


df = pd.DataFrame.from_records(data['messages'])
print(len(df))
df = df[df['type'] == 'message']
df = df[['date','date_unixtime', 'text', 'from']] #, 'text_entities'
df['text'] = df['text'].apply(clean_text_column)
df = df[df['text'] != '']
print(len(df))
df.to_csv('cleaned.csv')
df

#print(df.tail(300))

11014
9580


,date,date_unixtime,text,from
2,2021-08-23T20:55:03,1629741303,Visa Makes Their Grand Entrance Into the NFT A...,Watcher Guru
3,2021-08-26T03:07:52,1629936472,Have you seen our billboard in Times Square to...,Watcher Guru
4,2021-08-26T22:14:54,1630005294,Record breaking sale for Bored Ape,Watcher Guru
5,2021-09-02T20:58:54,1630605534,"This is our only Telegram, please be careful o...",Watcher Guru
6,2021-09-10T03:05:01,1631232301,Youtube Sensation MrBeast Reveals His Portfoli...,Watcher Guru
...,...,...,...,...
11008,2025-11-07T22:26:07,1762547167,Fed Governor Stephen Miran says crypto innovat...,Watcher Guru
11009,2025-11-07T22:36:03,1762547763,"Eric Trump says Bitcoin is going to hit $1,000...",Watcher Guru
11010,2025-11-08T04:33:55,1762569235,Supreme Court issues order allowing President ...,Watcher Guru
11011,2025-11-08T07:08:20,1762578500,"Binance Founder CZ says he was ""a little bit s...",Watcher Guru


In [4]:
file = 'result4.json'
with open(file) as cointelegraph:
    data = json.load(cointelegraph)


df_2 = pd.DataFrame.from_records(data['messages'])
print(len(df_2))
df_2 = df_2[df_2['type'] == 'message']
df_2 = df_2[['date','date_unixtime', 'text', 'from']] #, 'text_entities'
df_2['text'] = df_2['text'].apply(clean_text_column)
df_2 = df_2[df_2['text'] != '']
print(len(df_2))
df_2.to_csv('cleaned2.csv')
df_2


61616
58301


,date,date_unixtime,text,from
1,2016-08-25T23:47:56,1472158076,"3 Major Bitcoin Platforms in Philippines, Effi...",Cointelegraph
3,2016-08-26T12:24:48,1472203488,This Mobile Game is Teaching Mainstream Users ...,Cointelegraph
4,2016-08-26T13:41:10,1472208070,"Making Bitcoin Real: Portsmouth, N. H., Busine...",Cointelegraph
5,2016-08-26T15:29:37,1472214577,"Making Bitcoin Real: Portsmouth, N. H., Busine...",Cointelegraph
6,2016-08-26T16:17:10,1472217430,Elephant in Fintech Room: How ‘Big Four' Crash...,Cointelegraph
...,...,...,...,...
61611,2025-11-09T00:59:59,1762642799,FACTS: Everyone who said was dead has been wro...,Cointelegraph
61612,2025-11-09T01:59:59,1762646399,"ALERT: Thodex’s jailed CEO was found , ending ...",Cointelegraph
61613,2025-11-09T02:59:59,1762649999,NOW: $300B was wiped out from the market since...,Cointelegraph
61614,2025-11-09T04:00:00,1762653600,"NEW: Exchange Supply keeps dropping, per Crypt...",Cointelegraph


In [5]:
file = 'result5.json'
with open(file) as cryptonews:
    data = json.load(cryptonews)


df_3 = pd.DataFrame.from_records(data['messages'])
print(len(df_3))
df_3 = df_3[df_3['type'] == 'message']
df_3 = df_3[['date','date_unixtime', 'text', 'from']] #, 'text_entities'
df_3['text'] = df_3['text'].apply(clean_text_column)
df_3 = df_3[df_3['text'].str.len() > 3]
df_3 = df_3[df_3['text'] != 'JUST IN : Bitcoin hits']
print(len(df_3))
df_3.to_csv('cleaned3.csv')
df_3

19939
8732


,date,date_unixtime,text,from
1,2022-05-22T09:51:38,1653202298,Ripple Commits $100M to Invest in Carbon Marke...,Crypto News
2,2022-05-22T22:27:40,1653247660,Russian authorities have found an undercover B...,Crypto News
3,2022-05-23T06:50:00,1653277800,"Singapore court blocks BAYC sale, ruling NFTs ...",Crypto News
4,2022-05-23T10:18:58,1653290338,"The market is on the plus side today, most of ...",Crypto News
5,2022-05-23T16:11:56,1653311516,PayPal May Soon Accept All Forms Of Crypto Rev...,Crypto News
...,...,...,...,...
19918,2025-11-07T22:13:02,1762546382,1. ZEC 2. BDX 3. PENGU 4. ZERA 5. NEAR 6. ICP ...,Crypto News
19920,2025-11-08T02:00:11,1762560011,Index Value : 20 Sentiment : Extreme Fear BTC ...,Crypto News
19929,2025-11-08T15:00:11,1762606811,BTC : $101975ETH : $3406.0BNB : $993.07SOL : $...,Crypto News
19935,2025-11-08T22:13:02,1762632782,1. ZEC 2. ICP 3. ORE 4. GALA 5. PENGU 6. BTC 7...,Crypto News


In [6]:
file = 'result2.json'
with open(file) as unfolded:
    data = json.load(unfolded)


df_4 = pd.DataFrame.from_records(data['messages'])
print(len(df_4))
df_4 = df_4[df_4['type'] == 'message']
df_4 = df_4[['date','date_unixtime', 'text', 'from']] #, 'text_entities'
df_4['text'] = df_4['text'].apply(clean_text_column)
df_4 = df_4[df_4['text'].str.len() > 6]
df_4 = df_4[df_4['text'] != 'Follow on']
#df_4['date_unixtime'] = pd.to_numeric(df_4['date_unixtime'], downcast='integer', errors='coerce')
#df_4 =df_4[df_4['date_unixtime'] > 1629741303]
print(len(df_4))
df_4.to_csv('cleaned4.csv')
df_4

17110
16138


<unknown>:1: SyntaxWarning: invalid decimal literal


,date,date_unixtime,text,from
1,2018-06-15T07:09:20,1529035760,-7 (3 hours ago),unfolded.
2,2018-06-15T08:21:41,1529040101,"+6, -1 (3 hours ago)",unfolded.
3,2018-06-15T13:54:17,1529060057,"+7, -2 (an hour ago)",unfolded.
4,2018-06-15T14:09:34,1529060974,+8 (an hour ago),unfolded.
5,2018-06-15T14:35:04,1529062504,"+5, -4 (an hour ago)",unfolded.
...,...,...,...,...
17105,2025-11-07T02:38:24,1762475904,Jack Dorsey’s Block sees nearly $2 billion in ...,unfolded.
17106,2025-11-07T07:31:30,1762493490,Japan's regulator backs joint stablecoin proje...,unfolded.
17107,2025-11-07T14:30:36,1762518636,Tether moves nearly $100 million in Bitcoin to...,unfolded.
17108,2025-11-07T14:35:12,1762518912,Kazakhstan to launch $1 billion crypto reserve...,unfolded.


In [7]:
newdf = pd.concat([df, df_4], ignore_index=True, sort=False)

print(len(newdf))
newdf['date_unixtime'] = pd.to_numeric(newdf['date_unixtime'], downcast='integer', errors='coerce')
newdf =newdf[newdf['date_unixtime'] > 1546505895]

newdf.to_csv('headlines.csv')
newdf


25718


,date,date_unixtime,text,from
0,2021-08-23T20:55:03,1629741303,Visa Makes Their Grand Entrance Into the NFT A...,Watcher Guru
1,2021-08-26T03:07:52,1629936472,Have you seen our billboard in Times Square to...,Watcher Guru
2,2021-08-26T22:14:54,1630005294,Record breaking sale for Bored Ape,Watcher Guru
3,2021-09-02T20:58:54,1630605534,"This is our only Telegram, please be careful o...",Watcher Guru
4,2021-09-10T03:05:01,1631232301,Youtube Sensation MrBeast Reveals His Portfoli...,Watcher Guru
...,...,...,...,...
25713,2025-11-07T02:38:24,1762475904,Jack Dorsey’s Block sees nearly $2 billion in ...,unfolded.
25714,2025-11-07T07:31:30,1762493490,Japan's regulator backs joint stablecoin proje...,unfolded.
25715,2025-11-07T14:30:36,1762518636,Tether moves nearly $100 million in Bitcoin to...,unfolded.
25716,2025-11-07T14:35:12,1762518912,Kazakhstan to launch $1 billion crypto reserve...,unfolded.
